# 03 - Estimating sales and revenue

Steam does not publish sales. We estimate them from reviews: **1 review ≈ 20 to 60 sales** (Boxleiter method), then remove Steam's share (30 %).
The functions are in `src/estimation.py`.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.cleaning import load_clean
from src.estimation import (REFUND_RATE, PRICE_PAID_RATIO, estimate_sales, net_revenue,
                            add_estimates)

# Number display: thousands separator and 2 decimals.
pd.options.display.float_format = '{:,.2f}'.format

## 1. Test the functions by hand

In [2]:
for n in [0, 10, 1000]:
    print(n, 'reviews ->', estimate_sales(n, 'low'), '/', estimate_sales(n), '/', estimate_sales(n, 'high'), 'sales')

0 reviews -> 0 / 0 / 0 sales
10 reviews -> 200 / 400 / 600 sales
1000 reviews -> 20000 / 40000 / 60000 sales


In [3]:
# Sanity check: 1,000 reviews, $10, mid scenario, without refunds or discounts.
# By hand: 1,000 x 40 = 40,000 sales; 40,000 x $10 x 0.70 = $280,000.
sales = estimate_sales(1000)
print(sales, 'sales;', net_revenue(sales, 10, refund_rate=0, price_paid_ratio=1), '$ of net revenue')

40000 sales; 280000.0 $ of net revenue


## 2. Apply to the dataset

In [4]:
games = load_clean('../data/games_clean.csv')
df = add_estimates(games)
df[['name', 'price', 'reviews', 'sales_low', 'sales_mid', 'sales_high', 'revenue_mid']].head()

,name,price,reviews,sales_low,sales_mid,sales_high,revenue_mid
0,Supipara - Chapter 1 Spring Has Come!,14.97,255,5100,10200,15300,"72,147.92"
1,Mystery Solitaire The Black Raven,4.99,24,480,960,1440,"2,263.46"
2,Armored Brigade II,39.99,130,2600,5200,7800,"98,255.43"
3,Hepta Beats,0.99,32,640,1280,1920,598.75
4,MUMBA IV: Egypt Jewels,0.59,25,500,1000,1500,278.77


## 3. Realistic assumptions: refunds and discounts

In [5]:
basic = add_estimates(games, refund_rate=0, price_paid_ratio=1)
print('Refunds:', REFUND_RATE, '| price paid / listed price:', PRICE_PAID_RATIO)
print('Median revenue without adjustment:', round(basic['revenue_mid'].median()), '$')
print('Median revenue with adjustments:', round(df['revenue_mid'].median()), '$')
print('Ratio:', round(df['revenue_mid'].sum() / basic['revenue_mid'].sum(), 3))

Refunds: 0.1 | price paid / listed price: 0.75
Median revenue without adjustment: 2505 $
Median revenue with adjustments: 1691 $
Ratio: 0.675


The two adjustments multiply every revenue by 0.9 x 0.75 ≈ 0.68: one third less.
**We keep the adjusted version**: refunds exist on Steam (less than 2 h of play, less than 14 days), and most games sell a large share of their copies during sales. Ignoring both would overestimate revenue, and therefore the odds of being profitable.

## 4. The 10 games with the highest estimated revenue

In [6]:
top10 = df.sort_values('revenue_mid', ascending=False).head(10)
top10[['name', 'release_year', 'price', 'reviews', 'sales_mid', 'revenue_mid']]

,name,release_year,price,reviews,sales_mid,revenue_mid
25854,Black Myth: Wukong,2024,59.99,1150098,46003920,"1,303,993,763.48"
66969,ELDEN RING,2022,59.98,1056677,42267080,"1,197,872,294.09"
40330,Cyberpunk 2077,2020,59.97,844921,33796840,"957,661,343.79"
3155,Rust,2018,39.98,1227784,49111360,"927,740,601.65"
34777,Baldur's Gate 3,2023,59.99,760662,30426480,"862,446,942.88"
56363,Red Dead Redemption 2,2019,59.96,733542,29341680,"831,282,070.25"
38228,HELLDIVERS™ 2,2024,39.99,1017635,40705400,"769,139,726.98"
23338,The Witcher 3: Wild Hunt,2015,39.90,835349,33413960,"629,945,034.39"
64716,DARK SOULS™ III,2016,59.99,413775,16551000,"469,142,646.52"
31917,DayZ,2018,49.98,430252,17210080,"406,425,504.74"


The ranking is credible: these are well-known global hits (Black Myth: Wukong, Elden Ring, Baldur's Gate 3, Helldivers 2...). The order of magnitude (several tens of millions of sales) is consistent with the figures announced for these games.

# Validating the estimation method

## 1. Comparison with the SteamSpy range
We remove the games with 0 reviews and those whose range is "0 - 20000" (too imprecise).

In [7]:
valid = df[(df['reviews'] > 0) & (df['owners_min'] > 0)].copy()
print(len(valid), 'games with a usable SteamSpy range')

inside = (valid['sales_mid'] >= valid['owners_min']) & (valid['sales_mid'] <= valid['owners_max'])
below = valid['sales_mid'] < valid['owners_min']
above = valid['sales_mid'] > valid['owners_max']
print('Inside the range:', round(100 * inside.mean(), 1), '%')
print('Below           :', round(100 * below.mean(), 1), '%')
print('Above           :', round(100 * above.mean(), 1), '%')

18950 games with a usable SteamSpy range
Inside the range: 14.3 %
Below           : 83.1 %
Above           : 2.6 %


In [8]:
# Same computation for the 3 scenarios, and for the full low-high interval
for scenario in ['low', 'mid', 'high']:
    col = valid[f'sales_{scenario}']
    share = ((col >= valid['owners_min']) & (col <= valid['owners_max'])).mean()
    print(scenario, ':', round(100 * share, 1), '% inside the range')
overlap = (valid['sales_high'] >= valid['owners_min']) & (valid['sales_low'] <= valid['owners_max'])
print('Low-high interval overlapping the range:', round(100 * overlap.mean(), 1), '%')

low : 3.5 % inside the range
mid : 14.3 % inside the range
high : 20.2 % inside the range
Low-high interval overlapping the range: 27.9 %


## 2. Calibrate: how many owners per review?

In [9]:
valid['owners_mid'] = (valid['owners_min'] + valid['owners_max']) / 2
valid['ratio'] = valid['owners_mid'] / valid['reviews']
print('Median owners / reviews ratio:', round(valid['ratio'].median(), 1))

Median owners / reviews ratio: 244.8


In [10]:
valid['price_band'] = pd.cut(valid['price'], bins=[0, 5, 10, 15, 20, 30, 1000],
                             labels=['<5$', '5-10$', '10-15$', '15-20$', '20-30$', '30$+'])
print(valid.groupby('price_band', observed=True)['ratio'].agg(['median', 'count']).round(1))
print(valid.groupby('release_year')['ratio'].agg(['median', 'count']).round(1))

            median  count
price_band               
<5$         810.80   8354
5-10$       265.20   4091
10-15$      141.10   2243
15-20$       98.60   2088
20-30$       74.00   1283
30$+         56.20    891
              median  count
release_year               
2015          215.50   1345
2016          241.40   1737
2017          225.10   1694
2018          200.00   1601
2019          158.00   1348
2020          250.00   1770
2021          625.00   2275
2022          472.30   2192
2023          284.60   2087
2024          222.60   2049
2025          184.90    852


## 3. Known games with publicly announced sales

In [11]:
# The ratio depends a lot on the size of the game: look at it by number of reviews
valid['review_band'] = pd.cut(valid['reviews'], bins=[0, 100, 1000, 10000, 100000000],
                              labels=['1-100', '101-1000', '1001-10000', '10000+'])
print(valid.groupby('review_band', observed=True)['ratio'].agg(['median', 'count']).round(1))

              median  count
review_band                
1-100       3,181.80   7110
101-1000      169.50   6888
1001-10000     71.90   3860
10000+         47.60   1092


**Reading**: the overall median ratio (≈ 245 owners per review) is very far from 40, but it is biased:
- we only keep games with 20,000 owners or more: a small game with 100 reviews necessarily falls "below" the range;
- SteamSpy owners also count gifted keys, bundles and free weekends, especially for games under $5 (ratio > 800);
- for better-measured games, the ratio drops to 72 owners per review (1,000 to 10,000 reviews) and 48 (more than 10,000 reviews). Since owners ≥ sales, this remains compatible with 40 to 60 sales per review.

**Decision**: we keep 20 / 40 / 60. Our estimate is therefore rather **prudent** (below SteamSpy for most games), which is the right direction of error for a tool that advises spending a budget.

In [12]:
# Official figures announced by the studios or their publishers.
# Warning: these are all-platform sales (except Valheim, PC only in 2022),
# compared with our Steam-only estimate, computed with the 2026 reviews.
known = pd.DataFrame([
    {'name': 'Valheim', 'official_sales': 10000000, 'date': 'April 2022, PC',
     'source': 'https://www.pcgamer.com/valheim-has-sold-over-10-million-copies'},
    {'name': 'Balatro', 'official_sales': 5000000, 'date': 'January 2025, all platforms',
     'source': 'https://www.gematsu.com/2025/01/balatro-sales-top-five-million'},
    {'name': 'Hollow Knight', 'official_sales': 15000000, 'date': 'August 2025, all platforms',
     'source': 'https://www.avclub.com/hollow-knight-hits-all-time-peak-player-count-sells-15-million-copies'},
])
# merge: add our estimates next to each known game (join on the name).
comparison = known.merge(df[['name', 'reviews', 'sales_low', 'sales_mid', 'sales_high']], on='name')
comparison['gap_mid_%'] = (100 * (comparison['sales_mid'] - comparison['official_sales'])
                           / comparison['official_sales']).round(0)
comparison[['name', 'date', 'official_sales', 'sales_low', 'sales_mid', 'sales_high', 'gap_mid_%']]

,name,date,official_sales,sales_low,sales_mid,sales_high,gap_mid_%
0,Valheim,"April 2022, PC",10000000,9996560,19993120,29989680,100.00
1,Balatro,"January 2025, all platforms",5000000,3071320,6142640,9213960,23.00
2,Hollow Knight,"August 2025, all platforms",15000000,8318920,16637840,24956760,11.00


**Reading**: for the 3 games, the official figure falls between our low and mid scenarios.
The mid scenario overestimates (from +10 % for Hollow Knight to +100 % for Valheim), but the comparisons are imperfect: Valheim kept selling after 2022, and the Balatro and Hollow Knight figures include consoles and mobile.
These three hits are heavily "reviewed" games (many reviews per buyer): for them, the low scenario is the most accurate. For a small game, the previous section suggests on the contrary that 40 is prudent.

**Summary**: our mid estimate falls inside the SteamSpy range for 14 % of games (and below the range for 83 %, so it is prudent); for 3 known games, the official figure lies between our low and mid scenarios.